In [1]:
from pathlib import Path
import re
from collections import defaultdict
import heapq
import pandas as pd
import matplotlib.pyplot as plt

CONTACTS_DIR = Path("contacts")
CONTACTS_FILE = CONTACTS_DIR / "contacts-all13pf-swapAtoms.txt"
HBOND_FILES = {
    "strict": CONTACTS_DIR / "hbonds-all13pf-loose-swapAtoms.txt",
}

for p in [CONTACTS_FILE, *HBOND_FILES.values()]:
    if not p.exists():
        raise FileNotFoundError(f"Missing expected input file: {p}")

CONTACTS_FILE, HBOND_FILES

(PosixPath('contacts/contacts-all13pf-swapAtoms.txt'),
 {'strict': PosixPath('contacts/hbonds-all13pf-loose-swapAtoms.txt')})

In [2]:
def parse_contact_atom(atom_text: str) -> dict:
    atom_text = atom_text.strip()
    m = re.match(
        r"^(?P<source_pdb>\S+)\s+#(?P<model>[^/\s]+)/(?P<chain>\S+)\s+"
        r"(?P<residue>\S+)\s+(?P<residue_number>-?\d+)\s+(?P<atom>\S+)"
        r"(?:\s+(?P<altloc>\(.*\)))?$",
        atom_text,
    )
    if not m:
        return {
            "source_pdb": None,
            "model": None,
            "chain": None,
            "residue": None,
            "residue_number": None,
            "atom": None,
            "altloc": None,
            "raw_atom": atom_text,
        }

    return {
        "source_pdb": m.group("source_pdb"),
        "model": m.group("model"),
        "chain": m.group("chain"),
        "residue": m.group("residue"),
        "residue_number": int(m.group("residue_number")),
        "atom": m.group("atom"),
        "altloc": m.group("altloc"),
        "raw_atom": atom_text,
    }


def parse_contacts_file(path: Path) -> pd.DataFrame:
    lines = path.read_text().splitlines()
    header_idx = next(i for i, ln in enumerate(lines) if "atom1" in ln and "atom2" in ln and "overlap" in ln)
    data_lines = lines[header_idx + 1 :]

    line_re = re.compile(
        r"^(?P<atom1>.+?)\s{2,}(?P<atom2>.+?)\s{2,}(?P<overlap>-?\d+(?:\.\d+)?)\s+(?P<distance>-?\d+(?:\.\d+)?)\s*$"
    )

    rows = []
    for ln in data_lines:
        ln = ln.strip()
        if not ln:
            continue

        m = line_re.match(ln)
        if not m:
            continue

        atom1_txt = m.group("atom1").strip()
        atom2_txt = m.group("atom2").strip()

        a1 = parse_contact_atom(atom1_txt)
        a2 = parse_contact_atom(atom2_txt)

        rows.append(
            {
                "source_pdb": a1["source_pdb"] or a2["source_pdb"],
                "atom1": atom1_txt,
                "atom2": atom2_txt,
                "overlap": float(m.group("overlap")),
                "distance": float(m.group("distance")),
                **{f"atom1_{k}": v for k, v in a1.items() if k != "raw_atom"},
                **{f"atom2_{k}": v for k, v in a2.items() if k != "raw_atom"},
            }
        )

    return pd.DataFrame(rows)


def parse_hbond_contact_line(line: str):
    parts = re.split(r"\s{2,}", line.strip())
    if len(parts) == 5:
        donor_txt, acceptor_txt, hydrogen_txt, d_a_txt, d_h_a_txt = parts
    elif len(parts) == 4:
        donor_txt, acceptor_txt, d_a_txt, d_h_a_txt = parts
        hydrogen_txt = None
    else:
        return None

    donor = parse_contact_atom(donor_txt)
    acceptor = parse_contact_atom(acceptor_txt)
    hydrogen = parse_contact_atom(hydrogen_txt) if hydrogen_txt and hydrogen_txt != "no hydrogen" else None

    def _to_float(value):
        if value in {None, "", "N/A"}:
            return None
        return float(value)

    return {
        "source_pdb": donor["source_pdb"] or acceptor["source_pdb"],
        "donor": donor_txt,
        "acceptor": acceptor_txt,
        "hydrogen": hydrogen_txt,
        "d_a_dist": _to_float(d_a_txt),
        "d_h_a_dist": _to_float(d_h_a_txt),
        **{f"donor_{k}": v for k, v in donor.items() if k != "raw_atom"},
        **{f"acceptor_{k}": v for k, v in acceptor.items() if k != "raw_atom"},
        **(
            {f"hydrogen_{k}": v for k, v in hydrogen.items() if k != "raw_atom"}
            if hydrogen is not None
            else {
                "hydrogen_source_pdb": None,
                "hydrogen_model": None,
                "hydrogen_chain": None,
                "hydrogen_residue": None,
                "hydrogen_residue_number": None,
                "hydrogen_atom": None,
                "hydrogen_altloc": None,
            }
        ),
    }


def parse_hbond_file(path: Path, flavor: str) -> pd.DataFrame:
    lines = path.read_text().splitlines()
    rows = []

    for ln in lines:
        if ".pdb #" not in ln:
            continue
        parsed = parse_hbond_contact_line(ln)
        if parsed is None:
            continue
        parsed["hbond_flavor"] = flavor
        parsed["hbond_file"] = path.name
        rows.append(parsed)

    return pd.DataFrame(rows)

In [3]:
contacts_df = parse_contacts_file(CONTACTS_FILE)

hbond_frames = [parse_hbond_file(path, flavor) for flavor, path in HBOND_FILES.items()]
hbonds_df = pd.concat(hbond_frames, ignore_index=True)

print(f"Contacts rows: {len(contacts_df):,}")
print(f"H-bond rows (all flavors combined): {len(hbonds_df):,}")
print(f"Source structures in contacts: {contacts_df['source_pdb'].nunique()}")
print(f"Source structures in hbonds: {hbonds_df['source_pdb'].nunique()}")

contacts_df.head()
contacts_df.to_csv("contacts-all13pf.csv", index=False)
hbonds_df.to_csv("hbonds-all13pf.csv", index=False)

Contacts rows: 97,921
H-bond rows (all flavors combined): 26,122
Source structures in contacts: 4
Source structures in hbonds: 4


In [4]:


INTERFACE_DEFS = [
    ("A", "B", "intradimer"),
    ("A", "C", "interdimer"),
    ("A", "D", "lateral_alpha"),
    ("B", "E", "lateral_beta"),
    ("C", "F", "lateral_beta"),
]


def residue_node(chain, residue, residue_number):
    # Requested behavior: treat water as chainless and numberless.
    if residue == "HOH":
        return "HOH"
    return f"{chain}:{int(residue_number)}"


def parse_node(node):
    if node == "HOH":
        return (None, None)
    chain, num = node.split(":")
    return chain, int(num)


def contact_category(atom1_atom, atom2_atom):
    def fam(a):
        if pd.isna(a):
            return "other"
        s = str(a).upper()
        if s.startswith(("N", "O", "S", "P", "M", 'B', 'F', 'A')):
            return "polar"
        if s.startswith(("C")):
            return "hydrophobic"
        return "other"

    f1, f2 = fam(atom1_atom), fam(atom2_atom)
    if f1 == "polar" and f2 == "polar":
        return "polar"
    if f1 == "hydrophobic" and f2 == "hydrophobic":
        return "hydrophobic"
    return "other"


def build_graph_for_source(df_source):
    adj = defaultdict(dict)

    for _, r in df_source.iterrows():
        n1 = residue_node(r["atom1_chain"], r["atom1_residue"], r["atom1_residue_number"])
        n2 = residue_node(r["atom2_chain"], r["atom2_residue"], r["atom2_residue_number"])
        if n1 == n2:
            continue

        distance = float(r["distance"])
        cat = contact_category(r["atom1_atom"], r["atom2_atom"])

        key = tuple(sorted((n1, n2)))
        if key not in adj[n1]:
            adj[n1][key] = {"distance": distance, "categories": {cat}}
        else:
            adj[n1][key]["distance"] = min(adj[n1][key]["distance"], distance)
            adj[n1][key]["categories"].add(cat)

        if key not in adj[n2]:
            adj[n2][key] = {"distance": distance, "categories": {cat}}
        else:
            adj[n2][key]["distance"] = min(adj[n2][key]["distance"], distance)
            adj[n2][key]["categories"].add(cat)

    graph = defaultdict(list)
    edge_meta = {}
    seen = set()
    for u, edges in adj.items():
        for (a, b), meta in edges.items():
            if (a, b) in seen:
                continue
            seen.add((a, b))
            graph[a].append((b, meta["distance"]))
            graph[b].append((a, meta["distance"]))
            edge_meta[(a, b)] = meta

    return graph, edge_meta


def k_shortest_paths(graph, start, targets, k=3, max_hops=4):
    # Weighted by distance; also track number of connections (hops).
    results = []
    target_set = set(targets)
    pq = [(0.0, 0, [start])]

    while pq and len(results) < k:
        dist_sum, hops, path = heapq.heappop(pq)
        curr = path[-1]

        if hops > max_hops:
            continue

        if curr in target_set and curr != start:
            results.append((path, dist_sum, hops))
            continue

        for nxt, w in graph.get(curr, []):
            if nxt in path:
                continue
            heapq.heappush(pq, (dist_sum + w, hops + 1, path + [nxt]))

    return results


def build_axis_mapping(residue_numbers):
    vals = sorted(set(int(v) for v in residue_numbers))
    return {v: i for i, v in enumerate(vals)}, vals

In [5]:
# Build path-level interface dataframe (residue->residue with bridge information)
rows = []

for source_pdb, df_src in contacts_df.groupby("source_pdb"):
    graph, _ = build_graph_for_source(df_src)

    for chain_x, chain_y, iface in INTERFACE_DEFS:
        x_nodes = sorted(
            {
                residue_node(r["atom1_chain"], r["atom1_residue"], r["atom1_residue_number"])
                for _, r in df_src[df_src["atom1_chain"] == chain_x].iterrows()
                if r["atom1_residue"] != "HOH"
            }
            | {
                residue_node(r["atom2_chain"], r["atom2_residue"], r["atom2_residue_number"])
                for _, r in df_src[df_src["atom2_chain"] == chain_x].iterrows()
                if r["atom2_residue"] != "HOH"
            }
        )
        y_nodes = sorted(
            {
                residue_node(r["atom1_chain"], r["atom1_residue"], r["atom1_residue_number"])
                for _, r in df_src[df_src["atom1_chain"] == chain_y].iterrows()
                if r["atom1_residue"] != "HOH"
            }
            | {
                residue_node(r["atom2_chain"], r["atom2_residue"], r["atom2_residue_number"])
                for _, r in df_src[df_src["atom2_chain"] == chain_y].iterrows()
                if r["atom2_residue"] != "HOH"
            }
        )

        for start in x_nodes:
            paths = k_shortest_paths(graph, start, y_nodes, k=3, max_hops=4)
            for rank, (path_nodes, dist_sum, hops) in enumerate(paths, start=1):
                end = path_nodes[-1]
                _, x_resnum = parse_node(start)
                _, y_resnum = parse_node(end)

                rows.append(
                    {
                        "source_pdb": source_pdb,
                        "interface": iface,
                        "chain_x": chain_x,
                        "chain_y": chain_y,
                        "start_node": start,
                        "end_node": end,
                        "x_resnum": x_resnum,
                        "y_resnum": y_resnum,
                        "path_nodes": path_nodes,
                        "n_connections": hops,
                        "path_distance_sum": dist_sum,
                        "path_rank_for_target": rank,
                        "is_water_mediated": "HOH" in path_nodes,
                    }
                )

interface_paths_df = pd.DataFrame(rows)
print(f"Interface path rows: {len(interface_paths_df):,}")
interface_paths_df.head()

Interface path rows: 25,818


,source_pdb,interface,chain_x,chain_y,start_node,end_node,x_resnum,y_resnum,path_nodes,n_connections,path_distance_sum,path_rank_for_target,is_water_mediated
0,AlF3_6mer_swapAtoms.pdb,intradimer,A,B,A:1,B:57,1,57,"[A:1, HOH, B:57]",2,4.441,1,True
1,AlF3_6mer_swapAtoms.pdb,intradimer,A,B,A:1,B:503,1,503,"[A:1, HOH, B:503]",2,4.480,2,True
2,AlF3_6mer_swapAtoms.pdb,intradimer,A,B,A:1,B:264,1,264,"[A:1, HOH, B:264]",2,4.909,3,True
3,AlF3_6mer_swapAtoms.pdb,intradimer,A,B,A:10,B:57,10,57,"[A:10, HOH, B:57]",2,4.210,1,True
4,AlF3_6mer_swapAtoms.pdb,intradimer,A,B,A:10,B:503,10,503,"[A:10, HOH, B:503]",2,4.249,2,True


In [6]:
# Summary table comparable to original path outputs
interface_path_summary = (
    interface_paths_df.groupby(["source_pdb", "interface"], as_index=False)
    .agg(
        n_paths=("source_pdb", "size"),
        n_unique_pairs=("end_node", "nunique"),
        mean_path_distance=("path_distance_sum", "mean"),
        mean_connections=("n_connections", "mean"),
        n_water_mediated=("is_water_mediated", "sum"),
    )
    .sort_values(["source_pdb", "interface"])
)

interface_path_summary

,source_pdb,interface,n_paths,n_unique_pairs,mean_path_distance,mean_connections,n_water_mediated
0,AlF3_6mer_swapAtoms.pdb,interdimer,1296,28,5.138002,2.273920,1194
1,AlF3_6mer_swapAtoms.pdb,intradimer,1296,35,4.983816,2.252315,1162
2,AlF3_6mer_swapAtoms.pdb,lateral_alpha,1296,8,5.423348,2.318673,1248
3,AlF3_6mer_swapAtoms.pdb,lateral_beta,2586,12,5.565940,2.447796,2531
4,BeF3_6mer_swapAtoms.pdb,interdimer,1296,28,5.242346,2.264660,1183
5,BeF3_6mer_swapAtoms.pdb,intradimer,1296,34,5.176700,2.262346,1161
6,BeF3_6mer_swapAtoms.pdb,lateral_alpha,1296,10,5.376023,2.313272,1260
7,BeF3_6mer_swapAtoms.pdb,lateral_beta,2586,13,5.650483,2.486466,2522
8,GDP_13pf_6mer_fitted.pdb,interdimer,1293,27,5.696701,2.395978,1160
9,GDP_13pf_6mer_fitted.pdb,intradimer,1293,31,5.680415,2.382057,1120
